# 00 — Download to Google Drive (Colab)

Downloads split-0 labels + **mel shards 00–09** straight into Drive.

Folder: `/content/drive/MyDrive/MTG_Instrument`

Need ~**25 GB free on Drive**. Re-runs skip shards that already have `.shard_XX_done`.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


## Step 0 — Internet


In [ ]:
import socket
def check_internet(host="github.com", port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False
ONLINE = check_internet()
print("Internet:", ONLINE)
if not ONLINE:
    print("Turn on Internet in Colab, then re-run.")
!pip install -q tqdm


## Step 1 — Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


## Step 2 — Paths


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Step 3 — Download shards 00–09 onto Drive


In [ ]:
import subprocess, shutil

SHARDS = list(range(10))
BASE_URL = "https://cdn.freesound.org/mtg-jamendo/raw_30s/melspecs"
MEL_DIR.mkdir(parents=True, exist_ok=True)
print("Saving mels to", MEL_DIR)

def free_gb():
    u = shutil.disk_usage(str(MEL_DIR))
    print(f"Drive free: {u.free/1e9:.1f} GB")
    return u.free / 1e9

free_gb()
if not check_internet("cdn.freesound.org") and not check_internet():
    raise RuntimeError("No Internet — cannot download shards.")

for i in SHARDS:
    marker = MEL_DIR / f".shard_{i:02d}_done"
    if marker.exists():
        print(f"shard {i:02d} already on Drive — skip")
        continue
    if free_gb() < 3:
        raise RuntimeError(f"Not enough Drive space for shard {i:02d}")
    tar_name = f"raw_30s_melspecs-{i:02d}.tar"
    tar_path = MEL_DIR / tar_name
    url = f"{BASE_URL}/{tar_name}"
    print("Downloading", url)
    subprocess.check_call(["wget", "-q", "-O", str(tar_path), url])
    print("Extracting", tar_name)
    subprocess.check_call(["tar", "-xf", str(tar_path), "-C", str(MEL_DIR)])
    tar_path.unlink(missing_ok=True)
    marker.write_text("ok")
    print(f"shard {i:02d} saved")

n = len(list(MEL_DIR.rglob("*.npy")))
print("Total .npy on Drive:", n)
assert n > 0


## Step 4 — Summary


In [ ]:
summary = {
    "root": str(ROOT),
    "n_npy": len(list(MEL_DIR.rglob("*.npy"))),
    "shards": list(range(10)),
}
(RESULTS_DIR / "00_download_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Next: 01_preprocessing.ipynb in Colab, same Drive folder.")
